# pullgym — Proper-Scoring-Rule RLVR vs. Binary GRPO (Colab Pro)

This notebook runs the controlled RLVR calibration experiment for **`pullgym`** on a single **Colab Pro L4 (24 GB) or A100 (40 GB)** GPU at **$0 extra cost**.

### Experiment Design
- **Training split (8 families, seeds `100_000..`):** `line_fit`, `weighted_mean`, `counting_rate`, `decay_background`, `peak_background`, `low_count_rate`, `voltmeter_gain`, `underestimated_errors`.
- **Held-out OOD test split (4 unseen systematic/boundary families, dev seeds `0..`):** `errors_xy`, `correlated_norm`, `dead_time`, `no_signal`.
- **Controlled two-arm ablation (identical base model `Qwen3-4B`, prompts, and step budget):**
  1. **`ARM = "binary"` (Arm A — Point-Accuracy RLVR):** `+1.0` if valid quantiles and $|q_{50} - \theta| \le 1.0 \cdot \sigma_{\text{ref}}$, `0.0` otherwise (`-1.0` on crash/invalid).
  2. **`ARM = "log_score"` (Arm B — Proper Distributional RLVR):** clipped `log_score_reward(q, truth, ref_sigma)` (`[-10, +5]`, `-12.0` on crash/invalid).

### Colab Compute-Unit Saving Workflow
1. Start on an **L4 GPU** (~5 compute units/hr) with `SANITY_CHECK = True` (20 steps, ~10 min) to verify valid code execution rate (`valid_rate`) and non-zero within-group reward variance.
2. Set `SANITY_CHECK = False` (300–400 steps) on **L4** or **A100** (~12 units/hr, ~1.5–2.5 hrs). Checkpoints auto-save to Google Drive every 50 steps and auto-resume after disconnects.
3. Run once with `ARM = "binary"` and once with `ARM = "log_score"`, then run Cells 8–9 to compare **Base vs. Arm A vs. Arm B** on the 4 held-out OOD families.

In [ ]:
# Cell 1: Mount Google Drive and inspect GPU
from pathlib import Path
import subprocess
import torch

from google.colab import drive
drive.mount("/content/drive")

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime -> Change runtime type -> L4 or A100)"
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME} ({VRAM_GB:.1f} GB VRAM)")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"]).decode())

In [ ]:
# Cell 2: Install pullgym + Unsloth + vLLM
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

USE_DRIVE_REPO = True
DRIVE_REPO_PATH = Path("/content/drive/MyDrive/pullgym")
REPO_URL = "https://github.com/kekeandzeyu/pullgym.git"
# Do NOT name this /content/pullgym — /content is on sys.path[0] in Colab,
# which would shadow the installed `pullgym` package as a namespace package!
LOCAL_REPO = Path("/content/pullgym_repo")
if Path("/content/pullgym").exists():
    shutil.rmtree("/content/pullgym")

if USE_DRIVE_REPO and DRIVE_REPO_PATH.exists():
    if LOCAL_REPO.exists():
        shutil.rmtree(LOCAL_REPO)
    shutil.copytree(
        DRIVE_REPO_PATH,
        LOCAL_REPO,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".venv", ".git"),
    )
    for req_file in ("README.md", "LICENSE"):
        if not (LOCAL_REPO / req_file).exists():
            (LOCAL_REPO / req_file).write_text("pullgym\n", encoding="utf-8")
    REPO_DIR = LOCAL_REPO
else:
    if not LOCAL_REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(LOCAL_REPO), "pull", "--ff-only"], check=False)
    REPO_DIR = LOCAL_REPO

# Install pullgym editable + Unsloth/vLLM (pinned together to avoid TRL conflicts)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "pytest", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "vllm", "vllm-bnb-plugin"], check=True)

# Ensure the running Jupyter kernel immediately sees pullgym from REPO_DIR/src
src_dir = str(REPO_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
for mod in list(sys.modules):
    if mod == "pullgym" or mod.startswith("pullgym."):
        del sys.modules[mod]
importlib.invalidate_caches()
import pullgym.rl  # noqa: F401

# Verify pullgym runner & RL reward functions pass unit tests on this Colab instance
subprocess.run([sys.executable, "-m", "pytest", "-q", str(REPO_DIR / "tests" / "test_runner.py"), str(REPO_DIR / "tests" / "test_rl.py")], check=True)
print("Environment ready! pullgym loaded from:", pullgym.rl.__file__)

In [ ]:
# Cell 3: Experiment Configuration
from pathlib import Path

# Choose which arm to train in this run: "log_score" (Arm B) or "binary" (Arm A)
ARM = "log_score"  # change to "binary" for the controlled baseline arm

# Start with SANITY_CHECK = True (20 steps on L4) before launching the full 350-step run
SANITY_CHECK = True

BASE_MODEL = "unsloth/Qwen3-8B"
VARIANT = "calibrated"  # "calibrated" or "neutral"
ENABLE_THINKING = False  # Keep False so Qwen3 writes Python code directly without truncating inside <think>

# Dataset sizes
N_TRAIN_PER_FAMILY = 250   # 8 families x 250 = 2,000 training episodes
N_EVAL_OOD_PER_FAMILY = 25 # 4 OOD families x 25 = 100 held-out OOD episodes
N_EVAL_ID_PER_FAMILY = 15  # 8 train families x 15 = 120 in-distribution dev episodes

# Auto-tune VRAM settings for L4 (24GB) vs A100 (40GB+)
IS_A100 = VRAM_GB >= 35.0
LORA_RANK = 32 if IS_A100 else 16
MAX_PROMPT_LEN = 1536
MAX_COMPLETION_LEN = 2048 if IS_A100 else 1536
MAX_SEQ_LEN = MAX_PROMPT_LEN + MAX_COMPLETION_LEN
GPU_MEM_UTIL = 0.60 if IS_A100 else 0.50
NUM_GENERATIONS = 8 if IS_A100 else 4
GRAD_ACCUM = 4
MAX_STEPS = 20 if SANITY_CHECK else 100
SAVE_STEPS = 10 if SANITY_CHECK else 50

DRIVE_ROOT = Path("/content/drive/MyDrive/pullgym_rlvr")
RUN_NAME = f"{ARM}_{'sanity' if SANITY_CHECK else 'full'}"
OUTPUT_DIR = DRIVE_ROOT / "checkpoints" / RUN_NAME
LORA_DIR = DRIVE_ROOT / "loras" / RUN_NAME
EVAL_DIR = DRIVE_ROOT / "eval_runs"
for d in (OUTPUT_DIR, LORA_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Arm={ARM} | Sanity={SANITY_CHECK} | Steps={MAX_STEPS} | Gens={NUM_GENERATIONS} | MaxComp={MAX_COMPLETION_LEN}")
print(f"Checkpoints -> {OUTPUT_DIR}")

In [ ]:
# Cell 4: Build Train & Evaluation Datasets (8 train families vs. 4 OOD systematic families)
from datasets import Dataset
from pullgym.rl import (
    OOD_TEST_FAMILIES,
    TRAIN_FAMILIES,
    build_rl_records,
)

print("Building training dataset...")
train_records = build_rl_records(
    TRAIN_FAMILIES,
    n_per_family=N_TRAIN_PER_FAMILY,
    seed_start=100_000,
    split="train",
    variant=VARIANT,
    shuffle=True,
    shuffle_seed=42,
)
train_dataset = Dataset.from_list(train_records)

print("Building held-out OOD & in-distribution dev evaluation datasets...")
eval_ood_records = build_rl_records(
    OOD_TEST_FAMILIES,
    n_per_family=N_EVAL_OOD_PER_FAMILY,
    seed_start=0,
    split="dev",
    variant=VARIANT,
    shuffle=False,
)
eval_id_records = build_rl_records(
    TRAIN_FAMILIES,
    n_per_family=N_EVAL_ID_PER_FAMILY,
    seed_start=0,
    split="dev",
    variant=VARIANT,
    shuffle=False,
)

print(f"Train episodes: {len(train_dataset)} across {len(TRAIN_FAMILIES)} families")
print(f"OOD eval episodes: {len(eval_ood_records)} across {OOD_TEST_FAMILIES}")
print(f"ID eval episodes: {len(eval_id_records)} across {TRAIN_FAMILIES}")
print("\n--- Sample User Prompt (truncated) ---")
print(train_records[0]["prompt"][1]["content"][:1200])

In [ ]:
# Cell 5: Load Base Model with Unsloth + vLLM Fast Inference
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=GPU_MEM_UTIL,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model and LoRA adapters initialized.")

In [ ]:
# Cell 6: Fast Batched vLLM Evaluator + Untrained Base Model Evaluation
from vllm import SamplingParams
from pullgym.report import report
from pullgym.rl import evaluate_completions_to_trials


def render_prompt(msgs):
    if isinstance(msgs, str):
        return msgs
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING
        )
    except TypeError:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def eval_with_vllm(
    records,
    job_id: str,
    model_label: str,
    lora_request=None,
    temperature: float = 0.7,
    skip_if_exists: bool = True,
):
    target_dir = EVAL_DIR / job_id
    if skip_if_exists and target_dir.exists() and len(list(target_dir.rglob("verifier/details.json"))) == len(records):
        print(f"Skipping {job_id}: already evaluated ({len(records)} trials on Drive).")
        print(report(target_dir, group_by="model,family,variant", truth_bins=1))
        return target_dir

    prompts = [render_prompt(r["prompt"]) for r in records]
    # Qwen3 non-thinking mode recommends temperature=0.7, top_p=0.8, top_k=20, presence_penalty=1.0
    sampling = SamplingParams(
        temperature=temperature,
        top_p=0.8,
        top_k=20,
        presence_penalty=1.0,
        seed=42,
        max_tokens=MAX_COMPLETION_LEN,
    )
    outputs = model.fast_generate(
        prompts,
        sampling_params=sampling,
        lora_request=lora_request,
    )
    completions = [o.outputs[0].text for o in outputs]
    job_dir = evaluate_completions_to_trials(
        records,
        completions,
        EVAL_DIR,
        job_id=job_id,
        model_name=model_label,
        timeout_sec=15.0,
        max_workers=8,
    )
    print(f"\n=== {model_label} ({job_id}) ===")
    print(report(job_dir, group_by="model,family,variant", truth_bins=1))
    return job_dir


# Evaluate untrained base model once (cached on Drive for future arms)
eval_with_vllm(
    eval_ood_records,
    job_id=f"base_ood_{VARIANT}",
    model_label="qwen3-4b-base",
)
eval_with_vllm(
    eval_id_records,
    job_id=f"base_id_{VARIANT}",
    model_label="qwen3-4b-base",
)

In [ ]:
# Cell 7: Train with GRPOTrainer (auto-resumes from Drive checkpoints)
import numpy as np
from trl import GRPOConfig, GRPOTrainer
from pullgym.rl import (
    completion_text,
    make_binary_reward_fn,
    make_log_score_reward_fn,
)

if ARM == "log_score":
    base_reward_fn = make_log_score_reward_fn(
        floor=-10.0,
        invalid_penalty=-12.0,
        timeout_sec=8.0,
        max_workers=8,
    )
    INVALID_VAL = -12.0
elif ARM == "binary":
    base_reward_fn = make_binary_reward_fn(
        k=1.0,
        invalid_penalty=-1.0,
        timeout_sec=8.0,
        max_workers=8,
    )
    INVALID_VAL = -1.0
else:
    raise ValueError(f"Unknown ARM={ARM!r}; expected 'log_score' or 'binary'")

_call_counter = {"n": 0}


def logged_reward_fn(completions, files_json, truth_json, family=None, **kwargs):
    rewards = base_reward_fn(
        completions, files_json=files_json, truth_json=truth_json, **kwargs
    )
    _call_counter["n"] += 1
    arr = np.asarray(rewards, dtype=float)
    valid_rate = float(np.mean(arr > INVALID_VAL))
    if _call_counter["n"] % 10 == 1:
        fam_str = family[0] if family else "?"
        print(
            f"[batch {_call_counter['n']:4d} | {fam_str:<20s}] "
            f"valid={valid_rate:.0%} mean={arr.mean():+.3f} std={arr.std():.3f} "
            f"min={arr.min():+.2f} max={arr.max():+.2f}",
            flush=True,
        )
    if _call_counter["n"] % 50 == 1:
        best_i = int(np.argmax(arr))
        print(f"--- Sample best completion (reward={arr[best_i]:+.3f}) ---")
        print(completion_text(completions[best_i])[:900])
        print("---------------------------------------------------", flush=True)
    return rewards


training_args = GRPOConfig(
    use_vllm=True,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=max(2, int(0.05 * MAX_STEPS)),
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    beta=0.04,
    logging_steps=5,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LEN,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=4,
    max_grad_norm=0.2,
    report_to="none",
    output_dir=str(OUTPUT_DIR),
)

rendered_train = Dataset.from_list([
    {**r, "prompt": render_prompt(r["prompt"])} for r in train_records
])

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[logged_reward_fn],
    args=training_args,
    train_dataset=rendered_train,
)

has_ckpt = any(OUTPUT_DIR.glob("checkpoint-*"))
print(f"Starting GRPO training (resume_from_checkpoint={has_ckpt})...")
trainer.train(resume_from_checkpoint=has_ckpt)

model.save_lora(str(LORA_DIR))
print(f"Saved trained LoRA adapter to {LORA_DIR}")

In [ ]:
# Cell 8: Evaluate Trained Arm (OOD & ID) and Print Combined Leaderboard
trained_label = f"qwen3-4b-{RUN_NAME}"
lora_req = model.load_lora(str(LORA_DIR))

eval_with_vllm(
    eval_ood_records,
    job_id=f"{RUN_NAME}_ood_{VARIANT}",
    model_label=trained_label,
    lora_request=lora_req,
    skip_if_exists=False,
)
eval_with_vllm(
    eval_id_records,
    job_id=f"{RUN_NAME}_id_{VARIANT}",
    model_label=trained_label,
    lora_request=lora_req,
    skip_if_exists=False,
)

print("\n================ COMBINED SUMMARY (ALL MODELS ON DRIVE) ================")
print(report(EVAL_DIR, group_by="model,variant", truth_bins=1))
print("\n================ PER-FAMILY BREAKDOWN ================")
print(report(EVAL_DIR, group_by="model,family,variant", truth_bins=1))

In [ ]:
# Cell 9: Plot OOD Calibration Comparison & Inspect Generated Python Code
import json
from collections import defaultdict
import matplotlib.pyplot as plt
from pullgym.report import load_episodes
from pullgym.rl import OOD_TEST_FAMILIES
from pullgym.stats import summarize

rows, missing = load_episodes(EVAL_DIR)
ood_rows = [r for r in rows if r["family"] in OOD_TEST_FAMILIES]

by_model = defaultdict(list)
for r in ood_rows:
    by_model[r["model"]].append(r)

models = sorted(by_model)
if models:
    stats = {m: summarize(by_model[m]) for m in models}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    x = range(len(models))
    axes[0].bar([i - 0.18 for i in x], [stats[m].get("cov68", 0) * 100 for m in models], width=0.35, label="68% cov")
    axes[0].bar([i + 0.18 for i in x], [stats[m].get("cov95", 0) * 100 for m in models], width=0.35, label="95% cov")
    axes[0].axhline(68, color="C0", linestyle="--", alpha=0.7)
    axes[0].axhline(95, color="C1", linestyle="--", alpha=0.7)
    axes[0].set_title("OOD Empirical Coverage (%)")
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(models, rotation=20, ha="right")
    axes[0].legend()

    axes[1].bar(list(x), [stats[m].get("pull_width_robust", 0) for m in models], color="C2")
    axes[1].axhline(1.0, color="black", linestyle="--", label="Calibrated (1.0)")
    axes[1].set_title("OOD Robust Pull Width (lower error bars -> >1)")
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels(models, rotation=20, ha="right")
    axes[1].legend()

    axes[2].bar(list(x), [stats[m].get("reward_mean", 0) for m in models], color="C3")
    axes[2].axhline(0.0, color="black", linestyle="--", label="Reference (0.0)")
    axes[2].set_title("OOD Mean Log-Score Reward")
    axes[2].set_xticks(list(x))
    axes[2].set_xticklabels(models, rotation=20, ha="right")
    axes[2].legend()

    plt.tight_layout()
    plot_path = DRIVE_ROOT / f"ood_calibration_{VARIANT}.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Saved chart to {plot_path}")

# Inspect one OOD episode transcript (e.g. dead_time seed 0) across saved runs
sample_task = "dead_time-dev-00000"
for tpath in sorted(EVAL_DIR.glob(f"*/{sample_task}/transcript.json")):
    job = tpath.parent.parent.name
    rew = json.loads((tpath.parent / "verifier" / "reward.json").read_text())
    tr = json.loads(tpath.read_text())
    print(f"\n=== {job} | {sample_task} | reward={rew.get('reward', float('nan')):+.3f} valid={rew.get('valid')} ===")
    print(tr[-1]["content"][:1200])

### Optional Cell 10: Low-Cost Frontier Model Baselines (`pullgym run-fast`)

Run commercial models via any OpenAI-compatible API directly into `EVAL_DIR` so they appear in the exact same `pullgym report` table:

```bash
# 1. Gemini via Google AI Studio's OpenAI-compatible endpoint (free/low-cost tier)
export OPENAI_API_KEY="$GEMINI_API_KEY"
pullgym run-fast \
  --base-url https://generativelanguage.googleapis.com/v1beta/openai \
  --model gemini-2.5-flash \
  --n 25 --variant calibrated --concurrency 6 \
  --out /content/drive/MyDrive/pullgym_rlvr/eval_runs --resume

# 2. DeepSeek (50% off-peak discount)
export OPENAI_API_KEY="$DEEPSEEK_API_KEY"
pullgym run-fast \
  --base-url https://api.deepseek.com/v1 \
  --model deepseek-chat \
  --n 25 --variant calibrated --concurrency 6 \
  --out /content/drive/MyDrive/pullgym_rlvr/eval_runs --resume

# 3. Tabulate everything together
pullgym report /content/drive/MyDrive/pullgym_rlvr/eval_runs --by model,variant
```